# 13v1 - A3 v2: widened slice-sampling window, cache rebuild

**Plan item:** the "`RSNA_WINDOW` widening" lead. Full design, evidence,
and the gate the fold-0 pilot (`14v1`) is measured against:
`docs/superpowers/specs/2026-08-31-a3v2-window-widening-design.md`
(approved after one Opus review pass).

**This is this project's first-ever execution of the vendored
`build_pixel_cache.py` script** (cells below are copied verbatim from
`data/raw/_reference_kernels/rsna-knee-500gb-to-11gib-cpu-pixel-cache.ipynb`,
cells 10/11/12/13/15/25 - unedited except the env-var cell, which
deliberately omits `RSNA_WINDOW`). The current cache is
`stevenleehans`'s own separately-published Kaggle Dataset Output, not
something this project built before - so this notebook's own Step 0
(below) diffs the new build's fingerprint against the *current* cache's
recorded one, field by field, before spending any real decode time.

**CPU-only, deliberate** (same as A3's own original build) - runs during
a GPU-quota lockout if needed. Real cost, per the source's own measured
table: ~64 min CPU-only for the full corpus at the *current* 224px/130mm
config; window width changes which slice indices get read, not how many,
so this run's cost should land close to that same figure (checked in
Step 2 below, not assumed).

**Needs on Kaggle:** the competition data attached (train_series.csv,
test_series.csv, the full DICOM tree), the *current* cache Dataset
attached (for the fingerprint-diff step only - not read for its
pixels), internet OFF is fine (no model download needed, CPU-only).

In [ ]:
# Cell 10 of the reference kernel, VERBATIM. Real top-level imports and
# the T0/log() helper every function below relies on.
"""RSNA knee: study-level 12-label pipeline.

Synthesised from two community notebooks, with the parts that were wrong or missing in
both replaced. What is inherited, and from where:

  from `rsna-knee-baseline-v1`   header-recovered slot scheme, constant-physical-scale
                                 sampling, laterality normalisation, uint8 cache,
                                 per-diagnosis slot attention, report-hash grouping
  from `rsna-knee-eda-to-2-5d`   protocol-only features as a legal test-time signal

What is new here is listed in `README.md`; the five that change results are grouped
K-fold instead of a single holdout, gold studies held out of their own fold so the
annotated reference is honest, anatomy-preserving augmentation, a backbone that refuses
to train from random initialisation silently, and rank fusion of the folds and the
protocol model.
"""

from __future__ import annotations

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F


T0 = time.time()


def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

## Env vars - the intended change (spec section 2.1)

Same crop/group config as the current cache. `RSNA_WINDOW` is
deliberately **not set** - `read_slot()` (below) falls through to its
own `PLANE_WINDOW` per-plane defaults instead of the narrow
`0.35,0.65` pin the current cache was built with. `RSNA_IMG` is also
left unset (defaults to 224) - resolution stays out of scope for this
plan (spec section 5).

In [ ]:
os.environ["RSNA_CROP_MM"] = "130"
os.environ["RSNA_N_GROUP_MAX"] = "3"
# RSNA_WINDOW intentionally NOT set - this is the entire change vs. the
# current cache. Do not add it back.
os.environ.pop("RSNA_WINDOW", None)  # defensive: guard against Kaggle env leakage

for k, v in os.environ.items():
    if k.startswith("RSNA_"):
        print(f"{k} = {v}")
print("RSNA_WINDOW:", os.environ.get("RSNA_WINDOW", "(not set - PLANE_WINDOW defaults apply)"))

## `find_root()` - cell 12 of the reference kernel, VERBATIM (the `find_dinov2` half is unused here, omitted)

In [ ]:
def find_root(explicit=None):
    if explicit is not None:
        p = Path(explicit)
        if (p / "test.csv").is_file():
            return p
        raise FileNotFoundError(f"no test.csv under {p}")
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        # last resort: two-level scan, because the mount is sometimes nested one deeper
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file():
                    return cand
    raise FileNotFoundError("competition mount not found")

## `CFG` / `SLOTS` - cell 11 of the reference kernel, VERBATIM

In [ ]:
class CFG:
    seed = 2026               # folds, target construction, text distillation

    train_seed = int(os.environ.get("RSNA_TRAIN_SEED", "2026"))

    img = int(os.environ.get("RSNA_IMG", "224"))

    crop_mm = float(os.environ.get("RSNA_CROP_MM", "160"))

    pad_short_fov = os.environ.get("RSNA_PAD_SHORT_FOV", "0") == "1"

    swa_epochs = int(os.environ.get("RSNA_SWA_EPOCHS", "0"))

    lat_from_geometry = os.environ.get("RSNA_LAT_GEOMETRY", "0") == "1"
    group = 3                 # slices per encoder input, stacked as the three channels
    n_group_max = int(os.environ.get("RSNA_N_GROUP_MAX", "3"))
    cache_budget_gb = 12.0
    ram_fraction = 0.45       # ceiling on the cache as a share of free RAM

    hdr_threads = 16
    pix_threads = 12

    n_folds = 5
    max_folds_to_run = int(os.environ.get("RSNA_FOLDS", "5"))

    epochs = int(os.environ.get("RSNA_EPOCHS", "12"))
    cycle_epochs = int(os.environ.get("RSNA_CYCLE_EPOCHS", "0"))
    batch_studies = 8
    lr_backbone = 8e-6
    lr_head = 1e-3
    weight_decay = 0.02
    unfreeze_last = 6
    eval_batch = 12
    time_budget = float(os.environ.get("RSNA_TIME_BUDGET", 8.0 * 3600))

    w_text = 0.35
    w_protocol = 0.10
    gold_weight = 3.0


SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]

SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]

SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


def seed_all(seed=CFG.seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print(f"CFG.img={CFG.img}, CFG.crop_mm={CFG.crop_mm}, CFG.n_group_max={CFG.n_group_max}, "
      f"SLOT_SCHEME={SLOT_SCHEME!r}, N_SLOT={N_SLOT}")
assert CFG.img == 224, f"expected img=224 (unchanged, resolution out of scope), got {CFG.img}"
assert CFG.crop_mm == 130.0, f"expected crop_mm=130 (unchanged), got {CFG.crop_mm}"
assert CFG.n_group_max == 3, f"expected n_group_max=3 (unchanged), got {CFG.n_group_max}"
assert SLOT_SCHEME == "recovered", f"expected the recovered (6-slot) scheme, got {SLOT_SCHEME!r}"

## Header pass + slot picking - cell 13 of the reference kernel, VERBATIM

In [ ]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            "ImageLaterality", "StudyDescription", "BodyPartExamined",
            "PatientPosition", "ImagePositionPatient"]


def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(root, split):
    base = Path(root) / split
    items = []
    if not base.is_dir():
        return pd.DataFrame()
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=CFG.hdr_threads) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header."""
    if df.empty:
        return df
    for t in HDR_TAGS:
        if t not in df.columns:
            df[t] = None

    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1 = desc.str.contains(_T1_RX)
    t2 = desc.str.contains(_T2_RX)
    pdw = desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


def pick_slots(series_df, plane_map):
    """One series per slot per study."""
    if series_df.empty:
        return {}
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            if fluid is not None:
                sel = sel & (g["fluid"] == fluid)
            cand = g[sel]
            if len(cand) == 0 and fluid is False:
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out

## Slice ordering, sampling, and cache build - cell 15 of the reference
kernel, VERBATIM. `PLANE_WINDOW` here (not read yet - `read_slot()`
checks `RSNA_WINDOW` first, unset in this run) is the whole point of
this notebook.

In [ ]:
ORDER_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "SliceLocation",
              "InstanceNumber"]

import threading

ORDER_STATS = {"geometry": 0, "slice_location": 0, "instance_number": 0, "filename": 0}
_ORDER_LOCK = threading.Lock()

PIX_STATS = {"crop_applied": 0, "crop_padded": 0, "crop_skipped_short_fov": 0,
             "crop_no_spacing": 0, "decode_failed": 0, "mono1_inverted": 0,
             "slices_read": 0}
_PIX_LOCK = threading.Lock()


def _bump(key):
    with _ORDER_LOCK:
        ORDER_STATS[key] += 1


def _bump_pix(key, n=1):
    with _PIX_LOCK:
        PIX_STATS[key] += n


def order_slices(directory, files):
    """Return `files` sorted by position along the stack, nearest-first."""
    if len(files) < 2:
        return list(files)

    positions, locations, instances = [], [], []
    normal = None
    for name in files:
        try:
            ds = pydicom.dcmread(os.path.join(directory, name), stop_before_pixels=True,
                                 force=True, specific_tags=ORDER_TAGS)
        except Exception:
            positions.append(None)
            locations.append(None)
            instances.append(None)
            continue
        pos = getattr(ds, "ImagePositionPatient", None)
        orient = getattr(ds, "ImageOrientationPatient", None)
        if normal is None and orient is not None and len(orient) == 6:
            try:
                row_dir = np.array([float(v) for v in orient[:3]])
                col_dir = np.array([float(v) for v in orient[3:]])
                normal = np.cross(row_dir, col_dir)
            except Exception:
                normal = None
        try:
            positions.append(np.array([float(v) for v in pos]) if pos is not None else None)
        except Exception:
            positions.append(None)
        loc = getattr(ds, "SliceLocation", None)
        locations.append(float(loc) if loc is not None else None)
        num = getattr(ds, "InstanceNumber", None)
        instances.append(float(num) if num is not None else None)

    def sorted_by(values, stat):
        _bump(stat)
        return [f for _, f in sorted(zip(values, files), key=lambda pair: pair[0])]

    if normal is not None and all(p is not None for p in positions):
        return sorted_by([float(p @ normal) for p in positions], "geometry")
    if all(loc is not None for loc in locations):
        return sorted_by(locations, "slice_location")
    if all(num is not None for num in instances):
        return sorted_by(instances, "instance_number")
    _bump("filename")
    return list(files)


PLANE_WINDOW = {"Sagittal": (0.10, 0.90), "Axial": (0.10, 0.90),
                "Coronal": (0.15, 0.85)}


def read_slot(rec, n_slice, out_size, plane=None, group=None):
    """`n_slice` slices from one series, physically ordered, at `out_size` pixels."""
    files, d, px = rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    group = CFG.group if group is None else group

    files = order_slices(d, files)

    lo_f, hi_f = PLANE_WINDOW.get(plane, (0.10, 0.90))
    if os.environ.get("RSNA_WINDOW"):
        lo_f, hi_f = (float(x) for x in os.environ["RSNA_WINDOW"].split(","))
    lo, hi = int(lo_f * (n - 1)), int(hi_f * (n - 1))
    if hi <= lo:
        lo, hi = 0, n - 1

    n_anchor = max(1, n_slice // group)
    anchors = (np.linspace(lo, hi, n_anchor).astype(int) if n_anchor > 1
               else np.array([(lo + hi) // 2]))

    idx = []
    for centre in anchors:
        start = int(np.clip(centre - group // 2, 0, max(0, n - group)))
        idx.extend(range(start, min(start + group, n)))
    while len(idx) < n_slice:
        idx.append(idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
            if str(getattr(ds, "PhotometricInterpretation", "")).strip() == "MONOCHROME1":
                a = a.max() - a
                _bump_pix("mono1_inverted")
            _bump_pix("slices_read")
        except Exception:
            a = None
            _bump_pix("decode_failed")
        planes.append(a)

    shp = next((p.shape for p in planes if p is not None), None)
    if shp is None:
        return None
    planes = [p if (p is not None and p.shape == shp) else np.zeros(shp, np.float32)
              for p in planes]
    vol = np.stack(planes)

    if px and np.isfinite(px) and px > 0:
        want = int(round(CFG.crop_mm / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
            _bump_pix("crop_applied")
        elif want >= min(h, w):
            if CFG.pad_short_fov:
                py, pxd = max(0, want - h), max(0, want - w)
                vol = np.pad(vol, ((0, 0), (py // 2, py - py // 2),
                                   (pxd // 2, pxd - pxd // 2)))
                _bump_pix("crop_padded")
            else:
                _bump_pix("crop_skipped_short_fov")
        else:
            _bump_pix("crop_skipped_short_fov")
    else:
        _bump_pix("crop_no_spacing")

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


_SIDE_RX = [
    ("R", re.compile(r"\b(right|rt|r_?knee|knee_?r|dexter|sağ|derech[ao]|rechts?|"
                     r"droite?|δεξ\w*)\b", re.I)),
    ("L", re.compile(r"\b(left|lt|l_?knee|knee_?l|sinister|sol|izquierd[ao]|links?|"
                     r"gauche|αριστερ\w*)\b", re.I)),
]

LAT_STATS = {"tag": 0, "image_laterality": 0, "text": 0, "geometry": 0, "unknown": 0,
             "geom_agree": 0, "geom_disagree": 0}


def _lat_from_text(*fields):
    blob = " ".join(str(f) for f in fields if f and str(f).lower() != "nan")
    blob = re.sub(r"[^\w\s]", " ", blob)
    hits = {side for side, rx in _SIDE_RX if rx.search(blob)}
    return hits.pop() if len(hits) == 1 else None


def _lat_from_geometry(ipp):
    try:
        x = float(str(ipp).split("|")[0])
    except (TypeError, ValueError):
        return None
    if abs(x) < 20.0:
        return None
    return "R" if x < 0 else "L"


def resolve_laterality(g):
    vals = [str(x).strip().upper() for x in g.get("Laterality", pd.Series(dtype=object))
            .dropna()]
    vals = [v[0] for v in vals if v and v[0] in ("L", "R")]

    geom = next((s for s in (_lat_from_geometry(v)
                             for v in g.get("ImagePositionPatient",
                                            pd.Series(dtype=object)).dropna()) if s), None)
    if vals and geom:
        _bump_lat("geom_agree" if geom == vals[0] else "geom_disagree")
    if vals:
        _bump_lat("tag")
        return vals[0]

    ivals = [str(x).strip().upper() for x in
             g.get("ImageLaterality", pd.Series(dtype=object)).dropna()]
    ivals = [v[0] for v in ivals if v and v[0] in ("L", "R")]
    if ivals:
        _bump_lat("image_laterality")
        return ivals[0]

    txt = _lat_from_text(*g.get("SeriesDescription", pd.Series(dtype=object)).tolist(),
                         *g.get("StudyDescription", pd.Series(dtype=object)).tolist(),
                         *g.get("BodyPartExamined", pd.Series(dtype=object)).tolist())
    if txt:
        _bump_lat("text")
        return txt

    if geom and CFG.lat_from_geometry:
        _bump_lat("geometry")
        return geom

    _bump_lat("unknown")
    return None


def _bump_lat(key):
    LAT_STATS[key] += 1


def normalise_laterality(img, plane, lat):
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


def available_ram_gb():
    try:
        import psutil
        return psutil.virtual_memory().available / 1024 ** 3
    except Exception:
        pass
    try:
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return None


def plan_cache(n_study):
    budget = CFG.cache_budget_gb
    free = available_ram_gb()
    if free is not None:
        safe = CFG.ram_fraction * free
        if safe < budget:
            log(f"cache budget trimmed {budget:.1f} -> {safe:.1f} GB "
                f"({free:.1f} GB free, keeping {1 - CFG.ram_fraction:.0%} headroom)")
            budget = safe
    per_slice = n_study * N_SLOT * CFG.img * CFG.img
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(CFG.n_group_max, afford // CFG.group))
    if groups < CFG.n_group_max:
        log(f"cache budget {budget:.1f} GB allows {groups} group(s) of "
            f"{CFG.group}, not {CFG.n_group_max}")
    return groups


def build_cache(slot_map, plane_map, lat_map, tag, n_group):
    cache_slices = CFG.group * n_group
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, cache_slices, CFG.img, CFG.img), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")

    ORDER_STATS.update({k: 0 for k in ORDER_STATS})
    PIX_STATS.update({k: 0 for k in PIX_STATS})

    chunk = 512
    done = 0
    with ThreadPoolExecutor(max_workers=CFG.pix_threads) as pool:
        for c0 in range(0, len(jobs), chunk):
            block = jobs[c0:c0 + chunk]
            imgs = pool.map(lambda j: read_slot(j[3], cache_slices, CFG.img, j[2]), block)
            for (st, k, plane, _), img in zip(block, imgs):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < chunk:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > CFG.time_budget:
                log(f"  {tag}: time budget reached during decode")
                break
    total = sum(ORDER_STATS.values()) or 1
    log(f"{tag}: slice ordering " + ", ".join(
        f"{k} {v / total:.1%}" for k, v in ORDER_STATS.items() if v))
    if ORDER_STATS["filename"] / total > 0.05:
        warnings.warn(
            f"{ORDER_STATS['filename'] / total:.0%} of series fell back to filename "
            "order, which is SOP UID order and therefore random. Those slots carry no "
            "depth structure.", RuntimeWarning, stacklevel=2)

    n_crop = sum(PIX_STATS[k] for k in
                 ("crop_applied", "crop_padded", "crop_skipped_short_fov",
                  "crop_no_spacing")) or 1
    log(f"{tag}: physical scale " + ", ".join(
        f"{k.replace('crop_', '')} {PIX_STATS[k]} ({PIX_STATS[k] / n_crop:.1%})"
        for k in ("crop_applied", "crop_padded", "crop_skipped_short_fov",
                  "crop_no_spacing") if PIX_STATS[k]))
    log(f"{tag}: slices read {PIX_STATS['slices_read']}, "
        f"decode failures {PIX_STATS['decode_failed']}, "
        f"MONOCHROME1 inverted {PIX_STATS['mono1_inverted']}")
    skipped = PIX_STATS["crop_skipped_short_fov"] / n_crop
    if skipped > 0.05:
        warnings.warn(
            f"{skipped:.0%} of slots had a field of view at or below crop_mm="
            f"{CFG.crop_mm:.0f}mm, so no physical normalisation was applied to them and "
            "their mm/px came from the acquisition. Set RSNA_PAD_SHORT_FOV=1 to pad "
            "instead.", RuntimeWarning, stacklevel=2)
    gc.collect()
    return studies, cache, mask

## Step 0: fingerprint diff against the CURRENT cache (spec section
2.1, first-review finding C3)

Runs the same header pass + slot-picking `main()` does internally, so
`fp` is a real inspectable dict before any decode happens - not parsed
from stdout. Diffs every field except `window` against the *current*
cache's own `cache_meta.json`. Any unexplained difference is a real
confound (spec's own explicit instruction: "resolve or explicitly
accept and document" before trusting section 4's comparison).

**EDIT** `CURRENT_CACHE_META` below if the currently-attached cache
Dataset mounts somewhere other than the path already used throughout
this project.

In [ ]:
import json

CURRENT_CACHE_META = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache/cache_meta.json")
print("CURRENT_CACHE_META:", CURRENT_CACHE_META, "exists:", CURRENT_CACHE_META.exists())
assert CURRENT_CACHE_META.exists(), "current cache's cache_meta.json not found - check the Dataset is attached"

root = find_root()
log(f"input root: {root}")
seed_all()

train_series = pd.read_csv(root / "train_series.csv")
test_series = pd.read_csv(root / "test_series.csv")
both = pd.concat([train_series, test_series])
plane_map = dict(zip(both["SeriesInstanceUID"], both["Anatomical_Plane"]))

log("header pass: train")
htr = annotate(walk(root, "train_series"))
log("header pass: test")
hte = annotate(walk(root, "test_series"))
for h in (htr, hte):
    if not h.empty:
        h["plane"] = h["SeriesInstanceUID"].map(plane_map)

slots = {"train": pick_slots(htr, plane_map), "test": pick_slots(hte, plane_map)}

n_group = CFG.n_group_max


def fingerprint(n_group):
    return {
        "img": CFG.img,
        "crop_mm": CFG.crop_mm,
        "pad_short_fov": CFG.pad_short_fov,
        "lat_from_geometry": CFG.lat_from_geometry,
        "group": CFG.group,
        "n_group": int(n_group),
        "window": os.environ.get("RSNA_WINDOW", "default"),
        "slots": [s[0] for s in SLOTS],
        "seed": CFG.seed,
    }


fp = fingerprint(n_group)
print("\nnew build fingerprint:", json.dumps(fp, indent=2, sort_keys=True))

current_meta = json.loads(CURRENT_CACHE_META.read_text())
print("\ncurrent cache fingerprint (relevant fields):",
      json.dumps({k: current_meta.get(k, "<absent>") for k in fp}, indent=2, sort_keys=True))

diff = {k: (current_meta.get(k, "<absent>"), v) for k, v in fp.items()
        if k != "window" and current_meta.get(k, "<absent>") != v}
print("\nUNEXPECTED DIFFERENCES (excluding window, which is meant to differ):")
if diff:
    for k, (old, new) in diff.items():
        print(f"  {k}: current={old!r} vs. new={new!r}")
    print("\n*** STOP and investigate before proceeding - per spec section 2.1, an "
          "unexplained field beyond `window` is a real confound, not something to ignore. ***")
else:
    print("  none - every field matches except window, as expected.")
print(f"\nwindow: current={current_meta.get('window', '<absent>')!r} vs. new={fp['window']!r}")

## Step 1: probe, corrected acceptance criterion (spec section 2.1,
first-review findings I1/I2)

The reference kernel's own markdown records a 40-study probe measuring
**9.0 slot-series/s** (projecting 0.71h) against a real sustained rate
of **6.25/s** (1.03h) - the probe itself runs ~44% optimistic. So the
right check here is **rate-to-rate** at the same `n_study=40`, not a
wall-clock comparison against the "~55 min" docstring figure (itself
corrected elsewhere in the same source to ~64.2 min CPU-only).

In [ ]:
def shard_name(tag, i, n):
    return tag if n == 1 else f"{tag}.s{i:02d}of{n:02d}"


def shard_studies(slot_map, i, n):
    studies = sorted(slot_map)
    bounds = np.linspace(0, len(studies), n + 1).astype(int)
    mine = studies[bounds[i]:bounds[i + 1]]
    return {s: slot_map[s] for s in mine}


def save_split(out, stem, studies, cache, mask):
    np.save(out / f"{stem}_cache.npy", cache)
    np.save(out / f"{stem}_mask.npy", mask)
    pd.DataFrame({"StudyInstanceUID": studies}).to_csv(
        out / f"{stem}_studies.csv", index=False)
    gb = cache.nbytes / 1024 ** 3
    log(f"{stem}: wrote {cache.shape} = {gb:.2f} GiB, "
           f"slot coverage {mask.mean():.1%}")
    return gb


def estimate_decode_time(slot_map, plane_map, lat_map, n_group, n_study, total_slot_series):
    studies = sorted(slot_map)[:n_study]
    sub = {s: slot_map[s] for s in studies}
    n_series = sum(len(v) for v in sub.values())
    t0 = time.time()
    build_cache(sub, plane_map, lat_map, "probe", n_group)
    dt = time.time() - t0
    rate = n_series / max(dt, 1e-6)
    proj = total_slot_series / max(rate, 1e-9)
    log(f"probe: {n_series} slot-series in {dt:.1f}s = {rate:.1f}/s")
    log(f"probe: {total_slot_series} slot-series projects to "
           f"{proj / 3600:.2f} h for the full split")
    return rate, proj


def lat_of(h):
    return {st: resolve_laterality(g)
            for st, g in h.groupby("StudyInstanceUID")} if not h.empty else {}


lats = {"train": lat_of(htr), "test": lat_of(hte)}
total_train_series = sum(len(v) for v in slots["train"].values())

PROBE_RATE, PROBE_PROJECTION_H = estimate_decode_time(
    slots["train"], plane_map, lats["train"], n_group, 40, total_train_series)

print(f"\nmeasured rate: {PROBE_RATE:.1f} slot-series/s vs. documented baseline 9.0/s "
      f"(the documented baseline was itself later found to run ~44% optimistic vs. a "
      f"real sustained 6.25/s - so a rate anywhere in the 6-10/s range is unremarkable; "
      f"a large deviation outside it is worth understanding before the full run)")
print(f"projected full-corpus decode: {PROBE_PROJECTION_H:.2f}h (train split)")

## Full build: train + test, 4 shards each (spec section 2.2)

**Only run this cell after reviewing Step 0's fingerprint diff (no
unexplained differences) and Step 1's probe rate (no large, unexplained
deviation).** Same `--shards 4` layout as the current cache - shard
count/size doesn't change with window width. `RSNA_TIME_BUDGET`
defaults to 8h in `CFG`; `main()` internally overrides it to 11h for a
real full-corpus run (see the reference kernel's own `main()` body,
reproduced below) so the decode isn't silently truncated.

In [ ]:
def main(splits="test,train", shards=4, shard="all", out_dir="/kaggle/working/cache"):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    CFG.time_budget = float(os.environ.get("RSNA_TIME_BUDGET", 11.0 * 3600))

    # slots/lats/plane_map/root already computed in Step 0's cell above -
    # reused here rather than recomputed, since nothing about them changes
    # between Step 0 and the full build.
    fp_now = fingerprint(n_group)
    log("fingerprint: " + json.dumps(fp_now, sort_keys=True))

    want = list(range(shards)) if shard == "all" else [int(x) for x in str(shard).split(",")]

    total_gb = 0.0
    written = {}
    for tag in splits.split(","):
        tag = tag.strip()
        if not tag or not slots.get(tag):
            log(f"{tag}: no slots, skipped")
            continue
        for i in want:
            stem = shard_name(tag, i, shards)
            if (out / f"{stem}_cache.npy").exists():
                log(f"{stem}: present, skipped")
                continue

            sub = shard_studies(slots[tag], i, shards)
            expect = sum(len(v) for v in sub.values())
            if not expect:
                log(f"{stem}: empty shard (no studies in this block)")
                written[stem] = {"studies": 0, "slot_series": 0}
                continue
            st, C, M = build_cache(sub, plane_map, lats[tag], stem, n_group)

            got = int(M.sum())
            if got < expect:
                raise RuntimeError(
                    f"{stem}: decoded {got} of {expect} slot-series "
                    f"({got / expect:.1%}). This shard is TRUNCATED and has NOT been "
                    "written. Re-run with more shards, or raise RSNA_TIME_BUDGET.")
            total_gb += save_split(out, stem, st, C, M)
            written[stem] = {"studies": len(st), "slot_series": got}
            del C, M

    meta_path = out / "cache_meta.json"
    prior = {}
    if meta_path.exists():
        prior = json.loads(meta_path.read_text())
        clash = {k: (prior.get(k), fp_now[k]) for k in fp_now
                 if k in prior and prior[k] != fp_now[k]}
        if clash:
            raise RuntimeError(f"config changed between shards: {clash}. Start a fresh --out.")
        if prior.get("shards", shards) != shards:
            raise RuntimeError(f"shards changed {prior['shards']} -> {shards}.")
        written = {**prior.get("splits", {}), **written}

    fp_now["shards"] = shards
    fp_now["splits"] = written
    fp_now["built_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    fp_now["decode_seconds"] = round(time.time() - t0, 1) + prior.get("decode_seconds", 0)
    meta_path.write_text(json.dumps(fp_now, indent=2, sort_keys=True))

    expect_shards = {shard_name(t.strip(), i, shards)
                     for t in splits.split(",") if slots.get(t.strip())
                     for i in range(shards)}
    missing = sorted(expect_shards - set(written))
    log(f"complete: {len(written)}/{len(expect_shards)} shards"
           if not missing else f"INCOMPLETE, still missing: {', '.join(missing)}")
    log(f"done in {(time.time() - t0) / 60:.1f} min, {total_gb:.2f} GiB written")
    if total_gb > 15:
        log(f"!! {total_gb:.1f} GiB may exceed the Kaggle output limit")
    return written


RESULT = main(splits="test,train", shards=4)
print("\nwritten:", json.dumps(RESULT, indent=2))

## Real output (fill in after running on Kaggle)

Paste back: Step 0's fingerprint diff output (any unexpected
differences?), Step 1's measured rate vs. the 9.0/s documented
baseline, the full build's `complete: N/N shards` line, and the final
`cache_meta.json` contents (especially `window: "default"`).